<a href="https://colab.research.google.com/github/AzureSolo/Land-suitability-project/blob/main/1_Data_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Run this cell with a Python runtime only.
# Switch back to R before running any cell below.
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:

# 1_Data_Cleaning.ipynb
# Project: land_suitability_project
# Purpose: Reformat and prepare the PLUTO files for analysis and futue data joins
# Reads: Primary_Land_Use_Tax_Lot_Output_(PLUTO)_20261118.csv
# Writes: clean_data/pluto_clean.csv

#switch runtime to R
setwd("/content/drive/MyDrive/land_suitability_project")


In [2]:
library(readr) # read_csv(), write_csv()
library(dplyr) # rename(), mutate(), filter(), left_join(), anti_join()
library(tidyr) # pivot_longer()


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [3]:
#read and review raw PLUTO data

pluto_raw <- read_csv(
  "raw_data/Primary_Land_Use_Tax_Lot_Output_(PLUTO)_20261118.csv",
  show_col_types = FALSE
)

# look at structure
head(pluto_raw,5)

Warning message:
“One or more parsing issues, call `problems()` on your data frame for details,
e.g.:
  dat <- vroom(...)
  problems(dat)”


borough,Tax block,Tax lot,community board,census tract 2010,cb2010,schooldist,council district,postcode,firecomp,⋯,transitzone,geom,basempdate,dcasdate,edesigdate,landmkdate,masdate,polidate,rpaddate,zoningdate
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,⋯,<chr>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>
QN,8786,42,413,1579.03,2005,26,23,11001,E251,⋯,Beyond the Greater Transit Zone,NA,NA,NA,NA,NA,NA,NA,NA,NA
QN,8786,46,413,1579.03,2005,26,23,11001,E251,⋯,Beyond the Greater Transit Zone,NA,NA,NA,NA,NA,NA,NA,NA,NA
QN,8786,51,413,1579.03,2005,26,23,11001,E251,⋯,Beyond the Greater Transit Zone,NA,NA,NA,NA,NA,NA,NA,NA,NA
QN,8798,2,413,1579.03,3006,26,23,11001,E251,⋯,Beyond the Greater Transit Zone,NA,NA,NA,NA,NA,NA,NA,NA,NA
QN,8798,3,413,1579.03,3006,26,23,11001,E251,⋯,Beyond the Greater Transit Zone,NA,NA,NA,NA,NA,NA,NA,NA,NA


In [4]:
names(pluto_raw)

[1] "borough"              "Tax block"            "Tax lot"             
  [4] "community board"      "census tract 2010"    "cb2010"              
  [7] "schooldist"           "council district"     "postcode"            
 [10] "firecomp"             "policeprct"           "healtharea"          
 [13] "sanitboro"            "sanitsub"             "address"             
 [16] "zonedist1"            "zonedist2"            "zonedist3"           
 [19] "zonedist4"            "overlay1"             "overlay2"            
 [22] "spdist1"              "spdist2"              "spdist3"             
 [25] "ltdheight"            "splitzone"            "bldgclass"           
 [28] "landuse"              "easements"            "ownertype"           
 [31] "ownername"            "lotarea"              "bldgarea"            
 [34] "comarea"              "resarea"              "officearea"          
 [37] "retailarea"           "garagearea"           "strgearea"           
 [40] "factryarea"           "otherarea"            "areasource"          
 [43] "numbldgs"             "numfloors"            "unitsres"            
 [46] "unitstotal"           "lotfront"             "lotdepth"            
 [49] "bldgfront"            "bldgdepth"            "ext"                 
 [52] "proxcode"             "irrlotcode"           "lottype"             
 [55] "bsmtcode"             "assessland"           "assesstot"           
 [58] "exempttot"            "yearbuilt"            "yearalter1"          
 [61] "yearalter2"           "histdist"             "landmark"            
 [64] "builtfar"             "residfar"             "commfar"             
 [67] "facilfar"             "affresfar"            "mnffar"              
 [70] "borocode"             "BBL"                  "condono"             
 [73] "tract2010"            "xcoord"               "ycoord"              
 [76] "latitude"             "longitude"            "zonemap"             
 [79] "zmcode"               "sanborn"              "taxmap"              
 [82] "edesignum"            "appbbl"               "appdate"             
 [85] "plutomapid"           "version"              "sanitdistrict"       
 [88] "healthcenterdistrict" "firm07_flag"          "pfirm15_flag"        
 [91] "dcpedited"            "notes"                "bct2020"             
 [94] "bctcb2020"            "mihopt1"              "mihopt2"             
 [97] "mihopt3"              "mihopt4"              "transitzone"         
[100] "geom"                 "basempdate"           "dcasdate"            
[103] "edesigdate"           "landmkdate"           "masdate"             
[106] "polidate"             "rpaddate"             "zoningdate"

In [5]:
#Rename fields
#As this dataset has so many fields, I am focusing on the ones I think I will be using.
#As I may need the other columns during the analysis, I am leaving them but not changing the name yet. Otherwise, I could remove columns with select()

pluto_clean <- pluto_raw %>%
  rename(
    # Parcel Info
    borough_name = borough,
    tax_block = `Tax block`,
    tax_lot = `Tax lot`,
    parcel_id = BBL,
    condominium_number = condono,

    # Location
    community_board = `community board`,
    census_tract_2010 = `census tract 2010`,
    census_block_2010 = cb2010,
    school_district = schooldist,
    city_council_district = `council district`,
    zip_code = postcode,
    fire_company = firecomp,
    police_precinct = policeprct,
    health_area = healtharea,
    sanitation_borough = sanitboro,
    sanitation_subdistrict = sanitsub,
    street_address = address,
    x_coordinate = xcoord,
    y_coordinate = ycoord,
    latitude = latitude,
    longitude = longitude,

    # Zoning Land Use
    zoning_district_1 = zonedist1,
    zoning_district_2 = zonedist2,
    zoning_district_3 = zonedist3,
    zoning_district_4 = zonedist4,
    zoning_overlay_1 = overlay1,
    zoning_overlay_2 = overlay2,
    special_district_1 = spdist1,
    special_district_2 = spdist2,
    special_district_3 = spdist3,
    limited_height_district = ltdheight,
    split_zoning_flag = splitzone,
    building_class = bldgclass,
    land_use= landuse,
    easement_count = easements,

    # Ownership
    owner_type = ownertype,
    owner_name = ownername,

    # Measurements
    lot_area_sqft = lotarea,
    building_area_sqft = bldgarea,
    commercial_area_sqft = comarea,
    residential_area_sqft = resarea,
    office_area_sqft = officearea,
    retail_area_sqft = retailarea,
    garage_area_sqft = garagearea,
    storage_area_sqft = strgearea,
    factory_area_sqft = factryarea,
    other_area_sqft = otherarea,

    # Building Characteristics
    number_of_buildings = numbldgs,
    number_of_floors = numfloors,
    residential_units = unitsres,
    total_units = unitstotal,
    lot_frontage_ft = lotfront,
    lot_depth_ft = lotdepth,
    building_frontage_ft = bldgfront,
    building_depth_ft = bldgdepth,

    # Value
    assessed_land_value = assessland,
    assessed_total_value = assesstot,
    exempt_value = exempttot,

    # Building Age
    year_built = yearbuilt,
    major_alteration_year_1 = yearalter1,
    major_alteration_year_2 = yearalter2,

    # Historic Info
    historic_district = histdist,
    landmark_status = landmark,


    # Floor Are Ratio (FAR)
    built_far = builtfar,
    residential_far = residfar,
    commercial_far = commfar,
    facility_far = facilfar,
    affordable_housing_far = affresfar,
    manufacturing_far = mnffar

  )

In [6]:
names(pluto_clean)

[1] "borough_name"            "tax_block"              
  [3] "tax_lot"                 "community_board"        
  [5] "census_tract_2010"       "census_block_2010"      
  [7] "school_district"         "city_council_district"  
  [9] "zip_code"                "fire_company"           
 [11] "police_precinct"         "health_area"            
 [13] "sanitation_borough"      "sanitation_subdistrict" 
 [15] "street_address"          "zoning_district_1"      
 [17] "zoning_district_2"       "zoning_district_3"      
 [19] "zoning_district_4"       "zoning_overlay_1"       
 [21] "zoning_overlay_2"        "special_district_1"     
 [23] "special_district_2"      "special_district_3"     
 [25] "limited_height_district" "split_zoning_flag"      
 [27] "building_class"          "land_use"               
 [29] "easement_count"          "owner_type"             
 [31] "owner_name"              "lot_area_sqft"          
 [33] "building_area_sqft"      "commercial_area_sqft"   
 [35] "residential_area_sqft"   "office_area_sqft"       
 [37] "retail_area_sqft"        "garage_area_sqft"       
 [39] "storage_area_sqft"       "factory_area_sqft"      
 [41] "other_area_sqft"         "areasource"             
 [43] "number_of_buildings"     "number_of_floors"       
 [45] "residential_units"       "total_units"            
 [47] "lot_frontage_ft"         "lot_depth_ft"           
 [49] "building_frontage_ft"    "building_depth_ft"      
 [51] "ext"                     "proxcode"               
 [53] "irrlotcode"              "lottype"                
 [55] "bsmtcode"                "assessed_land_value"    
 [57] "assessed_total_value"    "exempt_value"           
 [59] "year_built"              "major_alteration_year_1"
 [61] "major_alteration_year_2" "historic_district"      
 [63] "landmark_status"         "built_far"              
 [65] "residential_far"         "commercial_far"         
 [67] "facility_far"            "affordable_housing_far" 
 [69] "manufacturing_far"       "borocode"               
 [71] "parcel_id"               "condominium_number"     
 [73] "tract2010"               "x_coordinate"           
 [75] "y_coordinate"            "latitude"               
 [77] "longitude"               "zonemap"                
 [79] "zmcode"                  "sanborn"                
 [81] "taxmap"                  "edesignum"              
 [83] "appbbl"                  "appdate"                
 [85] "plutomapid"              "version"                
 [87] "sanitdistrict"           "healthcenterdistrict"   
 [89] "firm07_flag"             "pfirm15_flag"           
 [91] "dcpedited"               "notes"                  
 [93] "bct2020"                 "bctcb2020"              
 [95] "mihopt1"                 "mihopt2"                
 [97] "mihopt3"                 "mihopt4"                
 [99] "transitzone"             "geom"                   
[101] "basempdate"              "dcasdate"               
[103] "edesigdate"              "landmkdate"             
[105] "masdate"                 "polidate"               
[107] "rpaddate"                "zoningdate"

In [7]:
#Table size
dim(pluto_clean)
nrow(pluto_clean)
ncol(pluto_clean)

[1] 858284    108

[1] 858284

[1] 108

In [8]:
#Check for duplicate entries
n_distinct(pluto_clean$parcel_id)

sum(duplicated(pluto_clean$parcel_id))

[1] 858284

[1] 0

In [9]:
#Missing Data or Gaps

#Remove columns with no data at all

pluto_clean <- pluto_clean %>%
  select(where(~ !all(is.na(.))))

In [10]:
#Change blanks to NA

pluto_clean <- pluto_clean %>%
  mutate(
    across(
      where(is.character),
      ~na_if(.x, "")
    )
  )

In [11]:
#Check changes in table size
#orginal 858284 columns * 108 rows

dim(pluto_clean)
nrow(pluto_clean)
ncol(pluto_clean)


[1] 858284     93

[1] 858284

[1] 93

In [12]:
#Verify data types

glimpse(pluto_clean)

Rows: 858,284
Columns: 93
$ borough_name            <chr> "QN", "QN", "QN", "QN", "QN", "QN", "QN", "QN"…
$ tax_block               <dbl> 8786, 8786, 8786, 8798, 8798, 8798, 8798, 8798…
$ tax_lot                 <dbl> 42, 46, 51, 2, 3, 5, 7, 9, 11, 13, 16, 17, 19,…
$ community_board         <dbl> 413, 413, 413, 413, 413, 413, 413, 413, 413, 4…
$ census_tract_2010       <dbl> 1579.03, 1579.03, 1579.03, 1579.03, 1579.03, 1…
$ census_block_2010       <dbl> 2005, 2005, 2005, 3006, 3006, 3006, 3006, 3006…
$ school_district         <dbl> 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26…
$ city_council_district   <dbl> 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23…
$ zip_code                <dbl> 11001, 11001, 11001, 11001, 11001, 11001, 1100…
$ fire_company            <chr> "E251", "E251", "E251", "E251", "E251", "E251"…
$ police_precinct         <dbl> 105, 105, 105, 105, 105, 105, 105, 105, 105, 1…
$ health_area             <dbl> 2170, 2170, 2170, 2170, 2170, 2170, 2170, 2170…
$ sanitation_b

In [23]:
#Convert numeric columns to text/character
#some columns do not need to be numeric and will be changed accordingly

pluto_clean <- pluto_clean %>%
    mutate(
      across(
        c(
        tax_lot,
        community_board,
        census_tract_2010,
        census_block_2010,
        school_district,
        city_council_district,
        zip_code,
        police_precinct,
        health_area,
        sanitation_borough,
        land_use,
        year_built,
        major_alteration_year_1,
        major_alteration_year_2,
        parcel_id),
      as.character))


In [24]:
#Clean land_use so the use is clear
pluto_clean <- pluto_clean %>%
  mutate(
    land_use = case_when(
      as.character(land_use) == "1"  ~ "One and Two Family Buildings",
      as.character(land_use) == "2"  ~ "Multi-Family Walk-Up Residences",
      as.character(land_use) == "3"  ~ "Multi-Family Elevator Residences",
      as.character(land_use) == "4"  ~ "Mixed Residential and Commercial",
      as.character(land_use) == "5"  ~ "Commercial Uses",
      as.character(land_use) == "6"  ~ "Industrial and Manufacturing",
      as.character(land_use) == "7"  ~ "Transportation and Utility",
      as.character(land_use) == "8"  ~ "Public Facilities and Institutions",
      as.character(land_use) == "9"  ~ "Open Space and Recreation",
      as.character(land_use) == "10" ~ "Parking",
      as.character(land_use) == "11" ~ "Vacant Land",
      TRUE ~ as.character(land_use)
    )
  )


In [25]:
#Verify land use changes took

land_use_classes <- unique(pluto_clean$land_use)

land_use_classes

[1] "One and Two Family Buildings"       "Mixed Residential and Commercial"  
 [3] "Commercial Uses"                    "Multi-Family Walk-Up Residences"   
 [5] "Parking"                            "Transportation and Utility"        
 [7] "Public Facilities and Institutions" "Multi-Family Elevator Residences"  
 [9] "Vacant Land"                        "Industrial and Manufacturing"      
[11] "Open Space and Recreation"          NA

In [26]:
#Reorder so parcel_id (likely the join field) is up front

pluto_clean <- pluto_clean %>%
  relocate(parcel_id, .before = 1)
#check reordering

head(pluto_clean, 2)









parcel_id,borough_name,tax_block,tax_lot,community_board,census_tract_2010,census_block_2010,school_district,city_council_district,zip_code,⋯,firm07_flag,pfirm15_flag,dcpedited,bct2020,bctcb2020,mihopt1,mihopt2,mihopt3,mihopt4,transitzone
<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<dbl>,<dbl>,<lgl>,<dbl>,<dbl>,<lgl>,<lgl>,<lgl>,<lgl>,<chr>
4087860042,QN,8786,42,413,1579.03,2005,26,23,11001,⋯,NA,NA,NA,4157903,41579032009,NA,NA,NA,NA,Beyond the Greater Transit Zone
4087860046,QN,8786,46,413,1579.03,2005,26,23,11001,⋯,NA,NA,NA,4157903,41579032009,NA,NA,NA,NA,Beyond the Greater Transit Zone


In [28]:
#Produce pluto_clean into the clean_data folder

write.csv(pluto_clean,
  file = "clean_data/pluto_clean.csv",
  row.names = FALSE
)